In [ ]:
# ============================ 1) SETUP & CONFIG ============================
import os, sys, json, time, math, copy, random, csv, io, glob, zipfile, shutil, contextlib, warnings
from collections import defaultdict, OrderedDict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms.functional as TF
from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
try:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
except ImportError:
    os.system("pip install -q pycocotools")
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

# ----------------------------- USER SETTINGS -----------------------------
QUICK_TEST    = False            # True = tiny run to verify the whole notebook works end to end
DATA_ROOT     = None             # None = auto-detect under /kaggle/input (zip or folder). Or set a path.
OUT_DIR       = "/kaggle/working/outputs"
PRETRAINED    = True             # needs Internet ON in Kaggle (downloads torchvision COCO weights)
LOCAL_WEIGHTS = None             # optional: path to a fasterrcnn_resnet50_fpn_v2 COCO .pth if Internet is OFF
NUM_WORKERS   = 2
SEED_FULL     = 0
SEEDS         = [0, 1, 2]        # few-shot seeds (each seed samples a different K-shot support set)
K_LIST        = [10]             # support-set sizes to run (annotated instances per class), e.g. [5, 10]
BACKBONE_TRAIN_LAYERS = 5        # trainable ResNet stages for full-data / base training (1-5)
EXIF_FIX      = True             # decided automatically in the audit cell
DEDUP_MODE    = "report"         # "report" = only report near-duplicate leakage; "remove" = drop leaked val/test images
HASH_THR      = 10               # Hamming distance (of 256 bits) under which two images count as near-duplicates
INFER_TUNE    = True             # choose test-time image size / flip-TTA on the VALIDATION set (no retraining)
INFER_SIZES   = (896, 1024)
INFER_TTA     = (False, True)
A_USE_RFS     = True             # repeat-factor sampling: rarer classes are seen more often (full-data + base training)
A_RFS_T       = 0.5
SAVE_ERROR_EXAMPLES = True       # saves the test images with the most errors (error analysis for the paper)

# Experiment A: full-data Faster R-CNN
A_EPOCHS = 40; A_BATCH = 4; A_LR = 0.005; A_PATIENCE = 8; EMA_DECAY = 0.999
MIN_SIZES_TRAIN = (640, 704, 768, 832, 896)   # multi-scale training (upsampling helps small caps)
MAX_SIZE = 1333

# Experiment B stage 1: base training
BASE_EPOCHS = 20; BASE_BATCH = 4; BASE_LR = 0.005; BASE_PATIENCE = 6

# Experiment B stage 2: few-shot fine-tuning
FS_EPOCHS = 60; FS_BATCH = 4; FS_LR = 0.001; FS_BODY_LR_MULT = 0.1
FS_REPEAT = 2                    # passes over the tiny support set per epoch
FS_EVAL_EVERY = 5; FS_PATIENCE = 6
SYN_PROB = 0.5                   # probability that a support sample is replaced by a synthesized one
LAMBDA_HINT = 10.0               # weight of hint (distillation) loss   (not given in the paper: tune)
LAMBDA_TRI  = 0.5                # weight of triplet loss               (not given in the paper: tune)
TRI_MARGIN  = 0.3                # triplet margin                       (not given in the paper: tune)
COS_SCALE   = 20.0               # cosine classifier scale (TFA default)
RUN_SWEEP   = False              # optional small hyper-parameter sweep (see sweep cell)

if QUICK_TEST:
    SEEDS = [0]; K_LIST = [3]; INFER_SIZES = (896,); INFER_TTA = (False,)
    A_EPOCHS = 1; BASE_EPOCHS = 1; FS_EPOCHS = 2; FS_REPEAT = 1; FS_EVAL_EVERY = 1

# ------------------------------- constants -------------------------------
CLASS_NAMES = ["cap", "label", "not-crumbled", "no-cap", "crumbled"]   # labels 1..5 (0 = background)
NAME2LABEL  = {"cap": 1, "label": 2, "not-crumbled": 3, "notcrumbled": 3, "no-cap": 4, "nocap": 4, "crumbled": 5}
BASE_IDS    = [1, 2, 3]
NOVEL_IDS   = [4, 5]
NUM_CLASSES = len(CLASS_NAMES) + 1

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEV.type == "cuda"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "ckpt"), exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| device", DEV)
if DEV.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Turn on a GPU accelerator in the Kaggle notebook settings.")

In [ ]:
# ============================ 2) LOCATE / EXTRACT DATASET ============================
def locate_splits(root):
    found = {}
    for dp, dn, fn in os.walk(root):
        if "_annotations.coco.json" in fn:
            name = os.path.basename(dp).lower()
            name = {"val": "valid", "validation": "valid"}.get(name, name)
            found[name] = dp
    return found

search_roots = [DATA_ROOT] if DATA_ROOT else ["/kaggle/input"]
splits = {}
for r in search_roots:
    splits = locate_splits(r)
    if splits:
        break

if not splits:  # try unzipping any zip found
    zips = []
    for r in search_roots:
        zips += glob.glob(os.path.join(r, "**", "*.zip"), recursive=True)
    print("zip files found:", zips)
    for z in zips:
        dest = os.path.join("/kaggle/working/data", os.path.splitext(os.path.basename(z))[0])
        if not os.path.exists(dest):
            with zipfile.ZipFile(z) as zf:
                zf.extractall(dest)
        print("extracted", z, "->", dest)
    splits = locate_splits("/kaggle/working/data")

if "train" not in splits:
    raise RuntimeError(
        "Could not find '_annotations.coco.json' under a 'train' folder. Export the Roboflow dataset in "
        "'COCO' format (not YOLO), upload the zip as a Kaggle dataset, add it to this notebook, "
        "or set DATA_ROOT to the extracted folder. Found: " + str(splits))
print("Splits found:", splits)

In [ ]:
# ============================ 3) LOAD ANNOTATIONS & DATASET STATS ============================
def norm_name(n):
    return n.strip().lower().replace("_", "-").replace(" ", "-")

def load_records(split_dir):
    d = json.load(open(os.path.join(split_dir, "_annotations.coco.json")))
    cat_map, unknown = {}, []
    for c in d["categories"]:
        key = norm_name(c["name"])
        if key in NAME2LABEL:
            cat_map[c["id"]] = NAME2LABEL[key]
        else:
            unknown.append(c["name"])
    found_labels = set(cat_map.values())
    if found_labels != set(range(1, 6)):
        raise RuntimeError("Expected the 5 classes %s but this split has categories %s"
                           % (CLASS_NAMES, [c["name"] for c in d["categories"]]))
    per_img = defaultdict(list)
    for a in d["annotations"]:
        if a["category_id"] not in cat_map:
            continue
        x, y, w, h = a["bbox"]
        if w < 2 or h < 2:
            continue
        per_img[a["image_id"]].append((x, y, x + w, y + h, cat_map[a["category_id"]]))
    recs = []
    for im in d["images"]:
        anns = per_img.get(im["id"], [])
        boxes = np.array([a[:4] for a in anns], dtype=np.float32).reshape(-1, 4)
        labels = np.array([a[4] for a in anns], dtype=np.int64)
        recs.append(dict(image_id=int(im["id"]), path=os.path.join(split_dir, im["file_name"]),
                         w=int(im["width"]), h=int(im["height"]), boxes=boxes, labels=labels))
    return recs, unknown

train_recs, unk = load_records(splits["train"])
if unk: print("Ignored non-target categories:", unk)

if "valid" in splits:
    val_recs, _ = load_records(splits["valid"])
else:
    rng = random.Random(0); rng.shuffle(train_recs)
    n_val = max(1, int(0.15 * len(train_recs)))
    val_recs, train_recs = train_recs[:n_val], train_recs[n_val:]
    print("No 'valid' split found: carved 15% of train as validation.")

if "test" in splits:
    test_recs, _ = load_records(splits["test"])
else:
    test_recs = val_recs
    print("WARNING: no 'test' split found; using validation as test (results will be optimistic).")

if QUICK_TEST:
    def _take(recs, n):
        with_def = [r for r in recs if np.isin(r["labels"], NOVEL_IDS).any()]
        rest = [r for r in recs if not np.isin(r["labels"], NOVEL_IDS).any()]
        return (with_def[: n // 2] + rest[: n - n // 2])
    train_recs = _take(train_recs, 60); val_recs = _take(val_recs, 12); test_recs = _take(test_recs, 12)

def count_table(recs):
    c = np.zeros(len(CLASS_NAMES), int)
    for r in recs:
        for l in r["labels"]:
            c[l - 1] += 1
    return c

stats = pd.DataFrame({"train": count_table(train_recs), "valid": count_table(val_recs), "test": count_table(test_recs)},
                     index=CLASS_NAMES)
stats.loc["IMAGES"] = [len(train_recs), len(val_recs), len(test_recs)]
print(stats)
allb = np.concatenate([r["boxes"] for r in train_recs if len(r["boxes"])]) if train_recs else np.zeros((0, 4))
if len(allb):
    wh = np.stack([allb[:, 2] - allb[:, 0], allb[:, 3] - allb[:, 1]], 1)
    print("train box width/height (px): median %s, 5th pct %s" % (np.median(wh, 0), np.percentile(wh, 5, axis=0)))
print("image size example:", train_recs[0]["w"], "x", train_recs[0]["h"])
stats.to_csv(os.path.join(OUT_DIR, "dataset_class_counts.csv"))

def restrict_records(recs, keep_ids, drop_empty=False):
    out = []
    for r in recs:
        m = np.isin(r["labels"], keep_ids)
        if drop_empty and not m.any():
            continue
        rr = dict(r); rr["boxes"] = r["boxes"][m]; rr["labels"] = r["labels"][m]
        out.append(rr)
    return out

In [ ]:
# ============================ 3b) IMAGE AUDIT: EXIF orientation, size consistency, size report ============================
def _unique_splits():
    out = [("train", train_recs), ("valid", val_recs)]
    if test_recs is not val_recs:
        out.append(("test", test_recs))
    return out

def exif_orientation(path):
    try:
        with Image.open(path) as im:
            return im.getexif().get(274, 1), im.size
    except Exception:
        return None, None

def audit_split(name, recs):
    n_missing = n_exif = n_raw_mis = n_tr_mis = 0
    for r in recs:
        o, size = exif_orientation(r["path"]) if os.path.exists(r["path"]) else (None, None)
        if o is None:
            n_missing += 1
            continue
        tsize = (size[1], size[0]) if o in (5, 6, 7, 8) else size
        if o != 1:
            n_exif += 1
        if size != (r["w"], r["h"]):
            n_raw_mis += 1
        if tsize != (r["w"], r["h"]):
            n_tr_mis += 1
        r["_raw_size"] = size
        r["_t_size"] = tsize
    return dict(split=name, images=len(recs), unreadable_or_missing=n_missing, with_exif_rotation=n_exif,
                json_size_mismatch_raw=n_raw_mis, json_size_mismatch_after_exif=n_tr_mis)

audit_df = pd.DataFrame([audit_split(n, rc) for n, rc in _unique_splits()])
print(audit_df.to_string(index=False))
raw_total = int(audit_df["json_size_mismatch_raw"].sum())
after_total = int(audit_df["json_size_mismatch_after_exif"].sum())
EXIF_FIX = after_total <= raw_total
print(f"\nJSON-vs-file size mismatches: raw={raw_total}, after EXIF transpose={after_total} -> EXIF_FIX = {EXIF_FIX}")

# drop records whose image file is missing/unreadable (in place, so shared dict objects stay consistent)
for n, rc in _unique_splits():
    before = len(rc)
    rc[:] = [r for r in rc if "_raw_size" in r]
    if len(rc) != before:
        print(f"  removed {before - len(rc)} records with missing/unreadable images from {n}")

# make record width/height and boxes agree with the pixels that will actually be loaded
rescaled = 0
for n, rc in _unique_splits():
    for r in rc:
        if r.get("_norm"):
            continue
        W, H = r["_t_size"] if EXIF_FIX else r["_raw_size"]
        if (W, H) != (r["w"], r["h"]):
            sx, sy = W / r["w"], H / r["h"]
            if len(r["boxes"]):
                r["boxes"] = (r["boxes"] * np.array([sx, sy, sx, sy], dtype=np.float32)).astype(np.float32)
            r["w"], r["h"] = int(W), int(H)
            rescaled += 1
        r["_norm"] = True
print("records rescaled to match actual image size:", rescaled)
audit_df.to_csv(os.path.join(OUT_DIR, "image_audit.csv"), index=False)

# ---------------- image-size distribution ----------------
size_rows = []
for n, rc in _unique_splits():
    vc = pd.Series([f"{r['w']}x{r['h']}" for r in rc]).value_counts()
    for s, c in vc.items():
        size_rows.append(dict(split=n, size=s, images=int(c)))
size_df = pd.DataFrame(size_rows)
size_df.to_csv(os.path.join(OUT_DIR, "image_size_distribution.csv"), index=False)
print("\nMost common image sizes (original pixels):")
print(size_df.groupby("split").head(5).to_string(index=False))

# ---------------- box-size statistics (original pixels and effective size at evaluation resolution) ----------------
EVAL_SHORT = MIN_SIZES_TRAIN[-1]
box_rows = []
for ci, cname in enumerate(CLASS_NAMES):
    ws, hs, rel, eff = [], [], [], []
    for r in train_recs:
        m = r["labels"] == ci + 1
        sc = min(EVAL_SHORT / min(r["w"], r["h"]), MAX_SIZE / max(r["w"], r["h"]))
        for b in r["boxes"][m]:
            w, h = float(b[2] - b[0]), float(b[3] - b[1])
            ws.append(w); hs.append(h)
            rel.append(math.sqrt(w * h / (r["w"] * r["h"])))
            eff.append(math.sqrt(w * h) * sc)
    if ws:
        sq = np.sqrt(np.array(ws) * np.array(hs))
        box_rows.append(dict(cls=cname, role="novel" if (ci + 1) in NOVEL_IDS else "base", n_boxes=len(ws),
                             median_w_px=round(float(np.median(ws)), 1), median_h_px=round(float(np.median(hs)), 1),
                             median_rel_size=round(float(np.median(rel)), 3),
                             pct_small_lt32px=round(100 * float(np.mean(sq < 32)), 1),
                             median_size_at_eval_px=round(float(np.median(eff)), 1),
                             pct_small_lt32_at_eval=round(100 * float(np.mean(np.array(eff) < 32)), 1)))
box_df = pd.DataFrame(box_rows)
box_df.to_csv(os.path.join(OUT_DIR, "box_size_stats.csv"), index=False)
print("\nBox size statistics (train split):")
print(box_df.to_string(index=False))
print(f"\nPREPROCESSING USED: short side resized to {MIN_SIZES_TRAIN} px (random during training), "
      f"{EVAL_SHORT} px at evaluation (tuned on validation), max side {MAX_SIZE} px; ImageNet mean/std normalization "
      f"inside the model; EXIF correction = {EXIF_FIX}.")

In [ ]:
# ============================ 3c) NEAR-DUPLICATE / LEAKAGE CHECK ACROSS SPLITS ============================
# Uses a 256-bit difference hash. If validation/test images are near-copies of training images, the reported
# accuracy is inflated. Check the saved pair images before deciding (DEDUP_MODE = "remove" drops leaked val/test images).
def dhash(path, hs=16):
    im = Image.open(path)
    if EXIF_FIX:
        im = ImageOps.exif_transpose(im)
    im = im.convert("L").resize((hs + 1, hs), Image.BILINEAR)
    a = np.asarray(im, dtype=np.int16)
    return np.packbits((a[:, 1:] > a[:, :-1]).flatten())          # 32 bytes

def hash_split(recs):
    if not recs:
        return np.zeros((0, 32), np.uint8)
    return np.stack([dhash(r["path"]) for r in tqdm(recs, desc="hashing", leave=False)])

def hamming_matrix(A, B):
    out = np.zeros((len(A), len(B)), dtype=np.int32)
    for i in range(0, len(A), 64):
        x = np.bitwise_xor(A[i:i + 64, None, :], B[None, :, :])
        out[i:i + 64] = np.unpackbits(x, axis=2).sum(axis=2)
    return out

def near_pairs(HA, HB, thr):
    if len(HA) == 0 or len(HB) == 0:
        return []
    D = hamming_matrix(HA, HB)
    ii, jj = np.where(D <= thr)
    return [(int(i), int(j), int(D[i, j])) for i, j in zip(ii, jj)]

test_is_val = test_recs is val_recs
H_train = hash_split(train_recs)
H_val = hash_split(val_recs)
H_test = H_val if test_is_val else hash_split(test_recs)

pairs_vt = near_pairs(H_val, H_train, HASH_THR)
pairs_tt = [] if test_is_val else near_pairs(H_test, H_train, HASH_THR)
pairs_tv = [] if test_is_val else near_pairs(H_test, H_val, HASH_THR)
leak_val = sorted(set(i for i, _, _ in pairs_vt))
leak_test_train = sorted(set(i for i, _, _ in pairs_tt))
leak_test_val = sorted(set(i for i, _, _ in pairs_tv))

print(f"valid images with a near-duplicate in train : {len(leak_val)} / {len(val_recs)}")
if not test_is_val:
    print(f"test  images with a near-duplicate in train : {len(leak_test_train)} / {len(test_recs)}")
    print(f"test  images with a near-duplicate in valid : {len(leak_test_val)} / {len(test_recs)}")

rows = []
for (i, j, d) in pairs_vt:
    rows.append(dict(split_a="valid", path_a=val_recs[i]["path"], split_b="train", path_b=train_recs[j]["path"], hamming=d))
for (i, j, d) in pairs_tt:
    rows.append(dict(split_a="test", path_a=test_recs[i]["path"], split_b="train", path_b=train_recs[j]["path"], hamming=d))
for (i, j, d) in pairs_tv:
    rows.append(dict(split_a="test", path_a=test_recs[i]["path"], split_b="valid", path_b=val_recs[j]["path"], hamming=d))
leak_df = pd.DataFrame(rows, columns=["split_a", "path_a", "split_b", "path_b", "hamming"])
leak_df.to_csv(os.path.join(OUT_DIR, "leakage_report.csv"), index=False)

plots_dir = os.path.join(OUT_DIR, "plots"); os.makedirs(plots_dir, exist_ok=True)
if len(leak_df):
    show = leak_df.sort_values("hamming").head(6)
    fig, axes = plt.subplots(len(show), 2, figsize=(8, 4 * len(show)))
    axes = np.array(axes).reshape(len(show), 2)
    for k, (_, rw) in enumerate(show.iterrows()):
        for c, (p, s) in enumerate([(rw["path_a"], rw["split_a"]), (rw["path_b"], rw["split_b"])]):
            axes[k, c].imshow(ImageOps.exif_transpose(Image.open(p)).convert("RGB") if EXIF_FIX else Image.open(p).convert("RGB"))
            axes[k, c].set_title(f"{s} (hamming {rw['hamming']})", fontsize=9); axes[k, c].axis("off")
    plt.tight_layout(); plt.savefig(os.path.join(plots_dir, "leakage_pairs.png"), dpi=120); plt.show()
    print("Inspect the pairs above. If they are genuinely the same scene/object, set DEDUP_MODE = 'remove' and rerun from the top.")
else:
    print("No near-duplicates found at this threshold.")

if DEDUP_MODE == "remove" and len(leak_df):
    drop_val = set(leak_val)
    drop_test = set(leak_test_train) | set(leak_test_val)
    if drop_val and len(drop_val) <= 0.5 * len(val_recs):
        val_recs = [r for i, r in enumerate(val_recs) if i not in drop_val]
        print(f"removed {len(drop_val)} leaked validation images")
    elif drop_val:
        print("WARNING: more than 50% of validation images look leaked; NOT removing. The split itself may be leaky.")
    if test_is_val:
        test_recs = val_recs
    elif drop_test and len(drop_test) <= 0.5 * len(test_recs):
        test_recs = [r for i, r in enumerate(test_recs) if i not in drop_test]
        print(f"removed {len(drop_test)} leaked test images")
    elif drop_test:
        print("WARNING: more than 50% of test images look leaked; NOT removing. The split itself may be leaky.")
    print("final sizes -> train %d | valid %d | test %d" % (len(train_recs), len(val_recs), len(test_recs)))

In [ ]:
# ============================ 4) DATASETS, AUGMENTATION, SAMPLE SYNTHESIS ============================
def augment_instance(crop):
    c = crop.clone()
    if random.random() < 0.3:                       # RGB channel transform
        c = c[torch.randperm(3)]
    if random.random() < 0.7:                       # per-channel gain
        c = c * torch.empty(3, 1, 1).uniform_(0.8, 1.2)
    if random.random() < 0.5:                       # Gaussian noise
        c = c + torch.randn_like(c) * random.uniform(0.0, 0.06)
    if random.random() < 0.3:                       # salt-and-pepper noise
        amt = random.uniform(0.0, 0.02)
        r = torch.rand(c.shape[1:])
        c[:, r < amt / 2] = 0.0
        c[:, r > 1 - amt / 2] = 1.0
    return c.clamp(0, 1)

def synthesize(img, boxes, bg="zeros"):
    # Extreme R-CNN style instance-level synthesis: crop instances, augment, paste on a zero-padded canvas
    _, H, W = img.shape
    if bg == "zeros":
        canvas = torch.zeros_like(img)
    else:
        canvas = img.mean(dim=(1, 2), keepdim=True).expand_as(img).clone()
    placed, newb, keep = [], [], []
    order = list(range(len(boxes))); random.shuffle(order)
    for i in order:
        x1, y1, x2, y2 = [int(round(float(v))) for v in boxes[i]]
        x1 = max(0, x1); y1 = max(0, y1); x2 = min(W, x2); y2 = min(H, y2)
        w, h = x2 - x1, y2 - y1
        if w < 2 or h < 2:
            continue
        crop = augment_instance(img[:, y1:y2, x1:x2])
        pos = None
        for _ in range(30):
            nx = random.randint(0, W - w); ny = random.randint(0, H - h)
            if all(nx >= px2 or nx + w <= px1 or ny >= py2 or ny + h <= py1 for (px1, py1, px2, py2) in placed):
                pos = (nx, ny); break
        if pos is None:
            continue
        nx, ny = pos
        canvas[:, ny:ny + h, nx:nx + w] = crop
        placed.append((nx, ny, nx + w, ny + h)); newb.append([nx, ny, nx + w, ny + h]); keep.append(i)
    if not keep:
        return img, boxes, list(range(len(boxes)))
    return canvas, torch.tensor(newb, dtype=torch.float32), keep

class DetDataset(Dataset):
    def __init__(self, records, train=False, color_jitter=0.0, syn_prob=0.0, repeat=1):
        self.records = records; self.train = train; self.cj = color_jitter
        self.syn_prob = syn_prob; self.repeat = repeat
    def __len__(self):
        return len(self.records) * self.repeat
    def __getitem__(self, i):
        r = self.records[i % len(self.records)]
        pil = Image.open(r["path"])
        if EXIF_FIX:
            pil = ImageOps.exif_transpose(pil)
        img = TF.to_tensor(pil.convert("RGB"))
        boxes = torch.as_tensor(r["boxes"], dtype=torch.float32).reshape(-1, 4)
        labels = torch.as_tensor(r["labels"], dtype=torch.int64)
        if self.train:
            if self.syn_prob > 0 and len(boxes) > 0 and random.random() < self.syn_prob:
                img, boxes, keep = synthesize(img, boxes)
                labels = labels[torch.as_tensor(keep, dtype=torch.long)]
            if random.random() < 0.5:
                W = img.shape[-1]
                img = img.flip(-1)
                if len(boxes):
                    boxes = boxes.clone(); boxes[:, [0, 2]] = W - boxes[:, [2, 0]]
            if self.cj > 0:
                b = 1 + random.uniform(-self.cj, self.cj); c = 1 + random.uniform(-self.cj, self.cj)
                img = (((img - 0.5) * c + 0.5) * b).clamp(0, 1)
        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor([r["image_id"]])}
        return img, target

def collate(batch):
    return tuple(zip(*batch))

def rfs_weights(records, t=0.5, classes=None):
    # repeat-factor sampling (LVIS style): images holding rarer classes get sampled more often
    cls = list(classes) if classes else list(range(1, len(CLASS_NAMES) + 1))
    n = max(len(records), 1)
    freq = {c: sum(1 for r in records if (r["labels"] == c).any()) / n for c in cls}
    rc = {c: (max(1.0, math.sqrt(t / freq[c])) if freq[c] > 0 else 1.0) for c in cls}
    w = []
    for r in records:
        present = [rc[c] for c in cls if (r["labels"] == c).any()]
        w.append(max(present) if present else 1.0)
    print("RFS repeat factors:", {CLASS_NAMES[c - 1]: round(rc[c], 2) for c in cls})
    return w

def sample_kshot(records, k, classes, seed):
    # TFA-style K-shot sampling: for each class, collect images until exactly K instances of that class
    rng = random.Random(seed)
    entries = []
    for c in classes:
        cand = [r for r in records if (r["labels"] == c).any()]
        rng.shuffle(cand)
        total = 0
        for r in cand:
            m = r["labels"] == c
            n = int(m.sum())
            if total + n > k:
                continue
            e = dict(r); e["boxes"] = r["boxes"][m]; e["labels"] = r["labels"][m]
            entries.append(e); total += n
            if total == k:
                break
        if total < k:
            print("  WARNING: class %s has only %d/%d instances available for the support set" % (CLASS_NAMES[c - 1], total, k))
    return entries

val_loader  = DataLoader(DetDataset(val_recs),  batch_size=4, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate)
test_loader = DataLoader(DetDataset(test_recs), batch_size=4, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate)
print("datasets ready")

In [ ]:
# ============================ 5) MODELS ============================
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor, TwoMLPHead
from torchvision.models.detection.roi_heads import RoIHeads, fastrcnn_loss

def load_torchvision_detector():
    if PRETRAINED and LOCAL_WEIGHTS is None:
        try:
            return fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1,
                                              trainable_backbone_layers=BACKBONE_TRAIN_LAYERS)
        except Exception as e:
            raise RuntimeError("Could not download the pretrained weights (%s). Turn Internet ON in the Kaggle "
                               "notebook settings, or set LOCAL_WEIGHTS to a local .pth file." % e)
    det = fasterrcnn_resnet50_fpn_v2(weights=None, weights_backbone=None)
    if LOCAL_WEIGHTS:
        det.load_state_dict(torch.load(LOCAL_WEIGHTS, map_location="cpu"), strict=False)
    return det

# ---------------- Experiment A model: plain Faster R-CNN (ResNet-50 FPN v2) ----------------
def build_full_model():
    det = load_torchvision_detector()
    in_f = det.roi_heads.box_predictor.cls_score.in_features
    det.roi_heads.box_predictor = FastRCNNPredictor(in_f, NUM_CLASSES)
    det.transform.min_size = MIN_SIZES_TRAIN
    det.transform.max_size = MAX_SIZE
    det.roi_heads.score_thresh = 0.001
    det.roi_heads.detections_per_img = 100
    return det

# ---------------- Experiment B model: Extreme R-CNN components ----------------
class CosineLinear(nn.Module):
    def __init__(self, in_f, out_f, scale=20.0):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_f, in_f))
        nn.init.normal_(self.weight, std=0.01)
        self.scale = scale
    def forward(self, x):
        x = F.normalize(x.float(), dim=1, eps=1e-6)
        w = F.normalize(self.weight.float(), dim=1, eps=1e-6)
        return self.scale * (x @ w.t())

class DecoupledBoxHead(nn.Module):
    # classification tower = 2 FC layers (new); regression tower = pretrained conv-FC head
    def __init__(self, reg_head, in_ch=256 * 7 * 7, rep=1024):
        super().__init__()
        self.cls_head = TwoMLPHead(in_ch, rep)
        self.reg_head = reg_head
    def forward(self, x):
        return torch.cat([self.cls_head(x), self.reg_head(x)], dim=1)

class DecoupledPredictor(nn.Module):
    def __init__(self, in_ch, num_classes, scale=20.0):
        super().__init__()
        self.cls_score = CosineLinear(in_ch, num_classes, scale)
        self.bbox_pred = nn.Linear(in_ch, num_classes * 4)
        nn.init.normal_(self.bbox_pred.weight, std=0.001)
        nn.init.constant_(self.bbox_pred.bias, 0)
    def forward(self, x):
        d = x.shape[1] // 2
        return self.cls_score(x[:, :d]), self.bbox_pred(x[:, d:])

def batch_hard_triplet(feats, labels, margin=0.3, max_anchor=64, max_bg=64):
    # cosine triplet loss with batch-hard mining; anchors = foreground RoIs; negatives include other classes + background
    feats = F.normalize(feats.float(), dim=1)
    fg = (labels > 0).nonzero().squeeze(1)
    if fg.numel() < 2:
        return feats.sum() * 0.0
    if fg.numel() > max_anchor:
        fg = fg[torch.randperm(fg.numel(), device=feats.device)[:max_anchor]]
    bg = (labels == 0).nonzero().squeeze(1)
    if bg.numel() > max_bg:
        bg = bg[torch.randperm(bg.numel(), device=feats.device)[:max_bg]]
    idx = torch.cat([fg, bg])
    f, l = feats[idx], labels[idx]
    na = fg.numel()
    sim = f[:na] @ f.t()                                   # [na, M]
    same = l[:na, None] == l[None, :]
    eye = torch.zeros_like(same)
    eye[torch.arange(na, device=feats.device), torch.arange(na, device=feats.device)] = True
    pos_mask = same & ~eye
    neg_mask = ~same
    valid = pos_mask.any(1) & neg_mask.any(1)
    if valid.sum() == 0:
        return feats.sum() * 0.0
    hardest_pos = sim.masked_fill(~pos_mask, 2.0).min(1).values
    hardest_neg = sim.masked_fill(~neg_mask, -2.0).max(1).values
    return F.relu(margin + hardest_neg - hardest_pos)[valid].mean()

class ExtremeRoIHeads(RoIHeads):
    def __init__(self, *args, use_triplet=False, margin=0.3, **kw):
        super().__init__(*args, **kw)
        self.use_triplet = use_triplet
        self.margin = margin
    def forward(self, features, proposals, image_shapes, targets=None):
        labels = None; regression_targets = None
        if self.training:
            proposals, matched_idxs, labels, regression_targets = self.select_training_samples(proposals, targets)
        box_features = self.box_roi_pool(features, proposals, image_shapes)
        box_features = self.box_head(box_features)
        class_logits, box_regression = self.box_predictor(box_features)
        result, losses = [], {}
        if self.training:
            loss_classifier, loss_box_reg = fastrcnn_loss(class_logits, box_regression, labels, regression_targets)
            losses = {"loss_classifier": loss_classifier, "loss_box_reg": loss_box_reg}
            if self.use_triplet:
                d = box_features.shape[1] // 2
                losses["loss_triplet"] = batch_hard_triplet(box_features[:, :d], torch.cat(labels), margin=self.margin)
        else:
            boxes, scores, labs = self.postprocess_detections(class_logits, box_regression, proposals, image_shapes)
            for i in range(len(boxes)):
                result.append({"boxes": boxes[i], "labels": labs[i], "scores": scores[i]})
        return result, losses

def hint_loss(student_feats, teacher_feats, flags):
    # mean squared L2 distance between teacher and student FPN maps, only for images with base-class objects
    per_img = 0
    for k in student_feats:
        d = (student_feats[k].float() - teacher_feats[k].float()) ** 2
        per_img = per_img + d.mean(dim=(1, 2, 3))
    per_img = per_img / len(student_feats)
    flags = flags.to(per_img.device).float()
    return (per_img * flags).sum() / flags.sum().clamp(min=1.0)

class ExtremeFasterRCNN(FasterRCNN):
    def __init__(self, *args, **kw):
        super().__init__(*args, **kw)
        self.__dict__["_teacher"] = None          # kept out of state_dict / parameters on purpose
    def set_teacher(self, teacher):
        self.__dict__["_teacher"] = teacher
    def forward(self, images, targets=None, base_flags=None):
        original_image_sizes = [tuple(img.shape[-2:]) for img in images]
        images, targets = self.transform(images, targets)
        features = self.backbone(images.tensors)
        if isinstance(features, torch.Tensor):
            features = OrderedDict([("0", features)])
        proposals, proposal_losses = self.rpn(images, features, targets)
        detections, detector_losses = self.roi_heads(features, proposals, images.image_sizes, targets)
        if self.training:
            losses = {}
            losses.update(detector_losses); losses.update(proposal_losses)
            if self._teacher is not None and base_flags is not None:
                with torch.no_grad():
                    tf = self._teacher(images.tensors)
                losses["loss_hint"] = hint_loss(features, tf, base_flags)
            return losses
        return self.transform.postprocess(detections, images.image_sizes, original_image_sizes)

def build_extreme(num_classes, use_triplet=False, margin=0.3):
    det = load_torchvision_detector()
    box_head = DecoupledBoxHead(det.roi_heads.box_head)
    predictor = DecoupledPredictor(1024, num_classes, COS_SCALE)
    model = ExtremeFasterRCNN(det.backbone, num_classes=None,
                              rpn_anchor_generator=det.rpn.anchor_generator, rpn_head=det.rpn.head,
                              box_roi_pool=det.roi_heads.box_roi_pool, box_head=box_head, box_predictor=predictor,
                              min_size=MIN_SIZES_TRAIN, max_size=MAX_SIZE)
    model.roi_heads = ExtremeRoIHeads(
        box_roi_pool=det.roi_heads.box_roi_pool, box_head=box_head, box_predictor=predictor,
        fg_iou_thresh=0.5, bg_iou_thresh=0.5, batch_size_per_image=512, positive_fraction=0.25,
        bbox_reg_weights=None, score_thresh=0.001, nms_thresh=0.5, detections_per_img=100,
        use_triplet=use_triplet, margin=margin)
    return model

def load_base_into(model, base_state):
    # load everything except the predictor, then copy base-class rows into the expanded predictor
    sd = {k: v for k, v in base_state.items() if "box_predictor" not in k}
    missing, unexpected = model.load_state_dict(sd, strict=False)
    bad = [k for k in missing if "box_predictor" not in k]
    if bad or unexpected:
        print("WARNING: unexpected state-dict mismatch:", bad[:5], list(unexpected)[:5])
    nb = base_state["roi_heads.box_predictor.cls_score.weight"].shape[0]
    P = model.roi_heads.box_predictor
    with torch.no_grad():
        P.cls_score.weight[:nb] = base_state["roi_heads.box_predictor.cls_score.weight"].to(P.cls_score.weight.device)
        P.bbox_pred.weight[:nb * 4] = base_state["roi_heads.box_predictor.bbox_pred.weight"].to(P.bbox_pred.weight.device)
        P.bbox_pred.bias[:nb * 4] = base_state["roi_heads.box_predictor.bbox_pred.bias"].to(P.bbox_pred.bias.device)

def set_trainable(model, unfreeze):
    for p in model.parameters():
        p.requires_grad = False
    head, body = [], []
    for p in model.roi_heads.box_predictor.parameters():
        p.requires_grad = True; head.append(p)
    if unfreeze:
        for mod in (model.roi_heads.box_head,):
            for p in mod.parameters():
                p.requires_grad = True; head.append(p)
        for mod in (model.rpn.head, model.backbone.fpn, model.backbone.body.layer4):
            for p in mod.parameters():
                p.requires_grad = True; body.append(p)
    groups = [dict(params=head, lr=FS_LR)]
    if body:
        groups.append(dict(params=body, lr=FS_LR * FS_BODY_LR_MULT))
    return groups

def freeze_bn(model):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()

print("model code ready")

In [ ]:
# ============================ 6) METRICS (COCO AP + precision/recall/F1/FNR + confusion) ============================
def safe_nanmean(a):
    a = np.asarray(a, dtype=float)
    return float(np.nanmean(a)) if np.isfinite(a).any() else float("nan")

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + ab[None, :] - inter + 1e-9)

def build_coco_gt(records):
    images, anns, aid = [], [], 1
    for r in records:
        images.append(dict(id=int(r["image_id"]), width=int(r["w"]), height=int(r["h"])))
        for b, l in zip(r["boxes"], r["labels"]):
            w, h = float(b[2] - b[0]), float(b[3] - b[1])
            anns.append(dict(id=aid, image_id=int(r["image_id"]), category_id=int(l),
                             bbox=[float(b[0]), float(b[1]), w, h], area=w * h, iscrowd=0)); aid += 1
    cats = [dict(id=i + 1, name=n) for i, n in enumerate(CLASS_NAMES)]
    gt = COCO()
    gt.dataset = dict(images=images, annotations=anns, categories=cats)
    with contextlib.redirect_stdout(io.StringIO()):
        gt.createIndex()
    return gt

@torch.no_grad()
def predict(model, loader, tta=None):
    if tta is None:
        tta = bool(getattr(model, "_use_tta", False))
    model.eval()
    preds = {}
    for imgs, tgts in tqdm(loader, desc="predict", leave=False):
        imgs = [i.to(DEV, non_blocking=True) for i in imgs]
        with torch.autocast(device_type=DEV.type, enabled=USE_AMP):
            outs = model(imgs)
            if tta:
                outs_f = model([i.flip(-1) for i in imgs])
        if tta:
            merged = []
            for o, of, im in zip(outs, outs_f, imgs):
                W = im.shape[-1]
                bf = of["boxes"].float().clone()
                bf[:, [0, 2]] = W - of["boxes"].float()[:, [2, 0]]
                boxes = torch.cat([o["boxes"].float(), bf])
                scores = torch.cat([o["scores"].float(), of["scores"].float()])
                labels = torch.cat([o["labels"], of["labels"]])
                keep = torchvision.ops.batched_nms(boxes, scores, labels, 0.5)[:100]
                merged.append(dict(boxes=boxes[keep], scores=scores[keep], labels=labels[keep]))
            outs = merged
        for o, t in zip(outs, tgts):
            preds[int(t["image_id"].item())] = dict(
                boxes=o["boxes"].float().cpu().numpy().reshape(-1, 4),
                scores=o["scores"].float().cpu().numpy(),
                labels=o["labels"].cpu().numpy().astype(int))
    return preds

def coco_metrics(gt, preds):
    nc = len(CLASS_NAMES)
    present = set(a["category_id"] for a in gt.dataset["annotations"])
    ap = np.array([0.0 if (i + 1) in present else np.nan for i in range(nc)])
    ap50, ap75 = ap.copy(), ap.copy()
    dets = []
    for iid, p in preds.items():
        for b, s, l in zip(p["boxes"], p["scores"], p["labels"]):
            dets.append({"image_id": int(iid), "category_id": int(l),
                         "bbox": [float(b[0]), float(b[1]), float(b[2] - b[0]), float(b[3] - b[1])], "score": float(s)})
    if len(dets) > 0:
        with contextlib.redirect_stdout(io.StringIO()):
            dt = gt.loadRes(dets)
            E = COCOeval(gt, dt, "bbox")
            E.evaluate(); E.accumulate()
        prec = E.eval["precision"]                  # [T, R, K, A, M]
        for k, cid in enumerate(E.params.catIds):
            pk = prec[:, :, k, 0, 2]
            if (pk > -1).any():
                ap[cid - 1] = pk[pk > -1].mean()
                p50 = prec[0, :, k, 0, 2]; p75 = prec[5, :, k, 0, 2]
                ap50[cid - 1] = p50[p50 > -1].mean() if (p50 > -1).any() else np.nan
                ap75[cid - 1] = p75[p75 > -1].mean() if (p75 > -1).any() else np.nan
    bi = [i - 1 for i in BASE_IDS]; ni = [i - 1 for i in NOVEL_IDS]
    m = dict(mAP=safe_nanmean(ap), mAP50=safe_nanmean(ap50), mAP75=safe_nanmean(ap75),
             nAP=safe_nanmean(ap[ni]), nAP50=safe_nanmean(ap50[ni]), nAP75=safe_nanmean(ap75[ni]),
             bAP=safe_nanmean(ap[bi]), bAP50=safe_nanmean(ap50[bi]), bAP75=safe_nanmean(ap75[bi]))
    for i, c in enumerate(CLASS_NAMES):
        m["AP_" + c] = ap[i]; m["AP50_" + c] = ap50[i]; m["AP75_" + c] = ap75[i]
    return m

def prf_metrics(records, preds, conf=0.5, iou_thr=0.5):
    nc = len(CLASS_NAMES)
    tp = np.zeros(nc, int); fp = np.zeros(nc, int); fn = np.zeros(nc, int)
    cm = np.zeros((nc, nc + 1), int)                 # rows: GT class, cols: predicted class, last col = missed
    img_def_total = img_def_hit = img_nodef_total = img_false_alarm = 0
    for r in records:
        p = preds[r["image_id"]]
        k = p["scores"] >= conf
        pb, ps, pl = p["boxes"][k], p["scores"][k], p["labels"][k]
        o = np.argsort(-ps); pb, ps, pl = pb[o], ps[o], pl[o]
        gb, gl = r["boxes"], r["labels"]
        for ci in range(nc):
            c = ci + 1
            g = gb[gl == c]; d = pb[pl == c]
            if len(d) == 0:
                fn[ci] += len(g); continue
            if len(g) == 0:
                fp[ci] += len(d); continue
            M = iou_matrix(d, g); used = np.zeros(len(g), bool)
            for di in range(len(d)):
                row = M[di].copy(); row[used] = -1
                j = int(row.argmax())
                if row[j] >= iou_thr:
                    used[j] = True; tp[ci] += 1
                else:
                    fp[ci] += 1
            fn[ci] += int((~used).sum())
        if len(gb):
            M = iou_matrix(gb, pb); used_d = np.zeros(len(pb), bool)
            for gi in range(len(gb)):
                if len(pb) == 0:
                    cm[gl[gi] - 1, nc] += 1; continue
                row = M[gi].copy(); row[used_d] = -1
                j = int(row.argmax())
                if row[j] >= iou_thr:
                    used_d[j] = True; cm[gl[gi] - 1, pl[j] - 1] += 1
                else:
                    cm[gl[gi] - 1, nc] += 1
        has_def = bool(np.isin(gl, NOVEL_IDS).any()); pred_def = bool(np.isin(pl, NOVEL_IDS).any())
        if has_def:
            img_def_total += 1; img_def_hit += int(pred_def)
        else:
            img_nodef_total += 1; img_false_alarm += int(pred_def)
    m = {"conf": float(conf)}
    P = np.full(nc, np.nan); R = np.full(nc, np.nan); F1 = np.full(nc, np.nan)
    for i, c in enumerate(CLASS_NAMES):
        P[i] = tp[i] / (tp[i] + fp[i]) if (tp[i] + fp[i]) > 0 else (0.0 if (tp[i] + fn[i]) > 0 else np.nan)
        R[i] = tp[i] / (tp[i] + fn[i]) if (tp[i] + fn[i]) > 0 else np.nan
        F1[i] = (2 * P[i] * R[i] / (P[i] + R[i])) if (np.isfinite(P[i]) and np.isfinite(R[i]) and (P[i] + R[i]) > 0) else (0.0 if np.isfinite(R[i]) else np.nan)
        m["P_" + c] = P[i]; m["R_" + c] = R[i]; m["F1_" + c] = F1[i]
        m["FNR_" + c] = (1 - R[i]) if np.isfinite(R[i]) else np.nan
        m["TP_" + c] = int(tp[i]); m["FP_" + c] = int(fp[i]); m["FN_" + c] = int(fn[i])
    m["precision"] = safe_nanmean(P); m["recall"] = safe_nanmean(R); m["f1"] = safe_nanmean(F1)
    ni = [i - 1 for i in NOVEL_IDS]
    dtp, dfn = tp[ni].sum(), fn[ni].sum()
    m["FNR_defects"] = float(dfn / (dtp + dfn)) if (dtp + dfn) > 0 else float("nan")
    m["img_defect_recall"] = img_def_hit / img_def_total if img_def_total else float("nan")
    m["img_false_alarm"] = img_false_alarm / img_nodef_total if img_nodef_total else float("nan")
    return m, cm

def choose_conf(records, preds, grid=np.arange(0.10, 0.96, 0.05)):
    best_t, best_f = 0.5, -1.0
    for t in grid:
        f1 = prf_metrics(records, preds, conf=float(t))[0]["f1"]
        if np.isfinite(f1) and f1 > best_f:
            best_f, best_t = f1, float(t)
    return round(best_t, 2)

def evaluate_preds(records, gt, preds, conf=0.5):
    m = coco_metrics(gt, preds)
    m2, cm = prf_metrics(records, preds, conf=conf)
    m.update(m2)
    return m, cm

val_gt  = build_coco_gt(val_recs)
test_gt = build_coco_gt(test_recs)
def tune_inference(model, val_pack=None):
    # choose test-time resolution and flip-TTA on the validation set only (no retraining needed)
    vr, vg = val_pack if val_pack else (val_recs, val_gt)
    if not INFER_TUNE:
        model._use_tta = False
        return dict(size=MIN_SIZES_TRAIN[-1], tta=False)
    best_score, best_cfg = -1.0, None
    for sz in INFER_SIZES:
        for tta in INFER_TTA:
            model.transform.min_size = (sz,)
            model._use_tta = tta
            sc = coco_metrics(vg, predict(model, val_loader))["mAP50"]
            if np.isfinite(sc) and sc > best_score:
                best_score, best_cfg = sc, (sz, tta)
    sz, tta = best_cfg if best_cfg else (MIN_SIZES_TRAIN[-1], False)
    model.transform.min_size = (sz,)
    model._use_tta = tta
    return dict(size=sz, tta=tta, val_mAP50=best_score)

print("metrics ready")

In [ ]:
# ============================ 7) TRAINING ENGINE (live per-epoch output + CSV logging) ============================
STD_KEYS = ["loss_classifier", "loss_box_reg", "loss_objectness", "loss_rpn_box_reg"]
EPOCH_COLS = (["run", "seed", "stage", "epoch", "lr", "time_s", "train_loss", "loss_classifier", "loss_box_reg",
               "loss_objectness", "loss_rpn_box_reg", "loss_hint", "loss_triplet",
               "val_mAP", "val_mAP50", "val_mAP75", "val_nAP50", "val_bAP50", "val_precision", "val_recall", "val_f1"]
              + ["val_AP50_" + c for c in CLASS_NAMES])
BASE_T = torch.tensor(BASE_IDS)

def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=enabled)

class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad = False
        self.decay = decay; self.updates = 0
    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        d = min(self.decay, (1 + self.updates) / (10 + self.updates))
        msd = model.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
            else:
                v.copy_(msd[k])

def append_csv(path, row, cols):
    new = not os.path.exists(path)
    with open(path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow({c: row.get(c, "") for c in cols})

def fmt(v, n=3):
    return "  -  " if (v is None or (isinstance(v, float) and not np.isfinite(v))) else f"{v:.{n}f}"

def train_loop(model, train_loader, val_fn, epochs, groups, run, seed, stage, log_path,
               eval_every=1, use_ema=False, patience=None, extreme=False,
               lam_hint=0.0, lam_tri=0.0, warmup_iters=200, min_lr_frac=0.01, bn_eval=True):
    for g in groups:
        g["base_lr"] = g["lr"]
    opt = torch.optim.SGD(groups, lr=groups[0]["lr"], momentum=0.9, weight_decay=1e-4)
    scaler = make_scaler(USE_AMP)
    all_params = [p for g in groups for p in g["params"]]
    ema = ModelEMA(model, EMA_DECAY) if use_ema else None
    total_iters = max(1, epochs * len(train_loader))
    warm = min(warmup_iters, max(1, int(0.1 * total_iters)))
    it = 0; best = -1.0; best_state = None; bad_evals = 0; t_start = time.time()
    print(f"\n=== {run} | seed {seed} | {stage} | {epochs} epochs x {len(train_loader)} iters | "
          f"trainable params {sum(p.numel() for p in all_params):,} ===")
    for epoch in range(1, epochs + 1):
        model.train()
        if bn_eval:
            freeze_bn(model)
        t0 = time.time(); sums = defaultdict(float); n = 0; skipped = 0
        pbar = tqdm(train_loader, desc=f"{run} ep {epoch}/{epochs}", leave=False)
        for imgs, tgts in pbar:
            imgs = [i.to(DEV, non_blocking=True) for i in imgs]
            tgts = [{k: v.to(DEV) for k, v in t.items()} for t in tgts]
            if it < warm:
                s = (it + 1) / warm
            else:
                prog = (it - warm) / max(1, total_iters - warm)
                s = min_lr_frac + (1 - min_lr_frac) * 0.5 * (1 + math.cos(math.pi * prog))
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * s
            with torch.autocast(device_type=DEV.type, enabled=USE_AMP):
                if extreme:
                    flags = torch.stack([torch.isin(t["labels"], BASE_T.to(DEV)).any() for t in tgts]).float()
                    ld = model(imgs, tgts, base_flags=flags)
                else:
                    ld = model(imgs, tgts)
            total = sum(ld[k] for k in STD_KEYS)
            if "loss_hint" in ld:
                total = total + lam_hint * ld["loss_hint"]
            if "loss_triplet" in ld:
                total = total + lam_tri * ld["loss_triplet"]
            if not torch.isfinite(total):
                skipped += 1; opt.zero_grad(set_to_none=True); it += 1; continue
            opt.zero_grad(set_to_none=True)
            scaler.scale(total).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(all_params, 10.0)
            scaler.step(opt); scaler.update()
            if ema is not None:
                ema.update(model)
            sums["train_loss"] += float(total.item()); n += 1
            for k in STD_KEYS + ["loss_hint", "loss_triplet"]:
                if k in ld:
                    sums[k] += float(ld[k].item())
            it += 1
            pbar.set_postfix(loss=f"{sums['train_loss'] / max(n, 1):.3f}")
        pbar.close()
        row = dict(run=run, seed=seed, stage=stage, epoch=epoch, lr=opt.param_groups[0]["lr"])
        for k, v in sums.items():
            row[k] = v / max(n, 1)
        if skipped:
            print(f"   (skipped {skipped} non-finite iterations)")
        do_eval = (epoch % eval_every == 0) or (epoch == epochs)
        ev = None
        if do_eval:
            eval_model = ema.module if ema is not None else model
            ev = val_fn(eval_model)
            for k in ["mAP", "mAP50", "mAP75", "nAP50", "bAP50", "precision", "recall", "f1"]:
                row["val_" + k] = ev[k]
            for c in CLASS_NAMES:
                row["val_AP50_" + c] = ev["AP50_" + c]
            score = ev["mAP50"] if np.isfinite(ev["mAP50"]) else -1.0
            if score > best:
                best = score; bad_evals = 0
                best_state = {k: v.detach().cpu().clone() for k, v in eval_model.state_dict().items()}
            else:
                bad_evals += 1
        row["time_s"] = time.time() - t0
        append_csv(log_path, row, EPOCH_COLS)
        extra = ""
        if "loss_hint" in row:
            extra += f" hint {row['loss_hint']:.4f}"
        if "loss_triplet" in row:
            extra += f" tri {row['loss_triplet']:.4f}"
        line = (f"[{run}|s{seed}] ep {epoch:03d}/{epochs:03d} lr {row['lr']:.5f} | loss {row.get('train_loss', float('nan')):.4f} "
                f"(cls {row.get('loss_classifier', 0):.3f} box {row.get('loss_box_reg', 0):.3f} "
                f"obj {row.get('loss_objectness', 0):.3f} rpn {row.get('loss_rpn_box_reg', 0):.3f}{extra})")
        if ev is not None:
            line += (f" | VAL mAP50 {fmt(ev['mAP50'])} mAP {fmt(ev['mAP'])} nAP50 {fmt(ev['nAP50'])} bAP50 {fmt(ev['bAP50'])} "
                     f"P {fmt(ev['precision'])} R {fmt(ev['recall'])} F1 {fmt(ev['f1'])} | best {best:.3f}")
        line += f" | {row['time_s']:.0f}s"
        print(line, flush=True)
        if patience is not None and ev is not None and bad_evals >= patience:
            print(f"   early stopping (no val improvement for {patience} evaluations)")
            break
    final_model = ema.module if ema is not None else model
    if best_state is not None:
        final_model.load_state_dict(best_state)
    print(f"=== done {run} | best val mAP50 {best:.4f} | {(time.time() - t_start) / 60:.1f} min ===")
    return dict(model=final_model, best=best, train_minutes=(time.time() - t_start) / 60,
                trainable=sum(p.numel() for p in all_params))

# ---------------- final reporting (test set, confidence threshold tuned on validation) ----------------
RESULTS = []          # one row per (run, seed)
PERCLASS = []         # long-format per-class rows
CONFS = {}            # confusion matrices

def final_report(run, seed, model, kind, extra=None, val_pack=None, test_pack=None):
    vr, vg = val_pack if val_pack else (val_recs, val_gt)
    tr, tg = test_pack if test_pack else (test_recs, test_gt)
    inf = tune_inference(model, (vr, vg))
    print(f"   inference setting chosen on validation: short side {inf['size']} | flip-TTA {inf['tta']}")
    vp = predict(model, val_loader)
    conf = choose_conf(vr, vp)
    vm, _ = evaluate_preds(vr, vg, vp, conf)
    tp_ = predict(model, test_loader)
    tm, cm = evaluate_preds(tr, tg, tp_, conf)
    row = dict(run=run, kind=kind, seed=seed, conf_threshold=conf, infer_size=inf["size"], infer_tta=inf["tta"],
               val_mAP50=vm["mAP50"], val_mAP=vm["mAP"])
    row.update(tm)
    row.update(extra or {})
    RESULTS.append(row)
    for i, c in enumerate(CLASS_NAMES):
        PERCLASS.append(dict(run=run, kind=kind, seed=seed, cls=c, role="novel" if (i + 1) in NOVEL_IDS else "base",
                             AP=tm["AP_" + c], AP50=tm["AP50_" + c], AP75=tm["AP75_" + c],
                             precision=tm["P_" + c], recall=tm["R_" + c], f1=tm["F1_" + c], FNR=tm["FNR_" + c],
                             TP=tm["TP_" + c], FP=tm["FP_" + c], FN=tm["FN_" + c], conf_threshold=conf))
    CONFS[(run, seed)] = cm
    flush_results()
    print(f"[TEST] {run} seed {seed} @conf {conf}: mAP50 {fmt(tm['mAP50'])} mAP {fmt(tm['mAP'])} "
          f"nAP50 {fmt(tm['nAP50'])} bAP50 {fmt(tm['bAP50'])} P {fmt(tm['precision'])} R {fmt(tm['recall'])} "
          f"F1 {fmt(tm['f1'])} FNR(defects) {fmt(tm['FNR_defects'])}")
    return row

KIND_FILES = {"full_data": "full_data_final_metrics.csv",
              "base_training": "base_training_final_metrics.csv",
              "fewshot": "fewshot_final_metrics.csv"}

def flush_results():
    if RESULTS:
        df = pd.DataFrame(RESULTS)
        for kind, fn in KIND_FILES.items():
            d = df[df["kind"] == kind]
            if len(d):
                d.to_csv(os.path.join(OUT_DIR, fn), index=False)
    if PERCLASS:
        pd.DataFrame(PERCLASS).to_csv(os.path.join(OUT_DIR, "per_class_metrics.csv"), index=False)
    rows = []
    for (run, seed), cm in CONFS.items():
        for gi, gc in enumerate(CLASS_NAMES):
            for pj in range(len(CLASS_NAMES) + 1):
                rows.append(dict(run=run, seed=seed, gt_class=gc,
                                 pred_class=(CLASS_NAMES[pj] if pj < len(CLASS_NAMES) else "MISSED"), count=int(cm[gi, pj])))
    if rows:
        pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, "confusion_matrices.csv"), index=False)

def load_pil(path):
    im = Image.open(path)
    if EXIF_FIX:
        im = ImageOps.exif_transpose(im)
    return im.convert("RGB")

def save_error_examples(model, run_name, records, conf, n=12):
    import matplotlib.patches as patches
    ldr = DataLoader(DetDataset(records), batch_size=4, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate)
    preds = predict(model, ldr)
    scored = []
    for r in records:
        m, _ = prf_metrics([r], preds, conf=conf)
        scored.append((sum(m["FP_" + c] + m["FN_" + c] for c in CLASS_NAMES), r))
    scored.sort(key=lambda x: -x[0])
    top = [r for e, r in scored[:n] if e > 0]
    if not top:
        print("no errors to display for", run_name); return
    cols = 4; nrows = int(math.ceil(len(top) / cols))
    fig, axes = plt.subplots(nrows, cols, figsize=(5 * cols, 5 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, r in zip(axes, top):
        ax.imshow(load_pil(r["path"]))
        for b, l in zip(r["boxes"], r["labels"]):
            ax.add_patch(patches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, edgecolor="lime", linewidth=2))
            ax.text(b[0], b[1], CLASS_NAMES[l - 1], color="lime", fontsize=8, va="bottom")
        p = preds[r["image_id"]]; k = p["scores"] >= conf
        for b, sc, l in zip(p["boxes"][k], p["scores"][k], p["labels"][k]):
            ax.add_patch(patches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, edgecolor="red",
                                           linewidth=2, linestyle="--"))
            ax.text(b[0], b[3], f"{CLASS_NAMES[l - 1]} {sc:.2f}", color="red", fontsize=8, va="top")
        ax.set_title(os.path.basename(r["path"])[:32] + "  (green=GT, red=pred)", fontsize=8)
    os.makedirs(os.path.join(OUT_DIR, "plots"), exist_ok=True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "plots", "error_examples_%s.png" % run_name), dpi=110)
    plt.show()

print("training engine ready")

In [ ]:
# ============================ 8) EXPERIMENT A: FULL-DATA FASTER R-CNN ============================
LOG_A = os.path.join(OUT_DIR, "epoch_log_full_data.csv")
set_seed(SEED_FULL)
model_A = build_full_model().to(DEV)
trainable_A = [p for p in model_A.parameters() if p.requires_grad]
ds_A = DetDataset(train_recs, train=True, color_jitter=0.15)
_drop = len(train_recs) >= 2 * A_BATCH
if A_USE_RFS:
    sampler_A = WeightedRandomSampler(rfs_weights(train_recs, A_RFS_T), num_samples=len(ds_A), replacement=True)
    loader_A = DataLoader(ds_A, batch_size=A_BATCH, sampler=sampler_A, num_workers=NUM_WORKERS,
                          collate_fn=collate, drop_last=_drop)
else:
    loader_A = DataLoader(ds_A, batch_size=A_BATCH, shuffle=True, num_workers=NUM_WORKERS,
                          collate_fn=collate, drop_last=_drop)

def val_fn_all(m):
    return evaluate_preds(val_recs, val_gt, predict(m, val_loader), conf=0.5)[0]

res_A = train_loop(model_A, loader_A, val_fn_all, epochs=A_EPOCHS, groups=[dict(params=trainable_A, lr=A_LR)],
                   run="A_full_data", seed=SEED_FULL, stage="full", log_path=LOG_A, eval_every=1,
                   use_ema=True, patience=A_PATIENCE, extreme=False)
best_A = res_A["model"]
row_A = final_report("A_full_data", SEED_FULL, best_A, kind="full_data",
             extra=dict(train_minutes=res_A["train_minutes"], trainable_params=res_A["trainable"], K=-1,
                        train_images=len(train_recs)))
if SAVE_ERROR_EXAMPLES:
    save_error_examples(best_A, "A_full_data", test_recs, row_A["conf_threshold"])
torch.save(best_A.state_dict(), os.path.join(OUT_DIR, "ckpt", "A_full_data_best.pth"))
del model_A, best_A, res_A, loader_A
torch.cuda.empty_cache()

In [ ]:
# ============================ 9) EXPERIMENT B / STAGE 1: BASE TRAINING ============================
LOG_BASE = os.path.join(OUT_DIR, "epoch_log_base_training.csv")
train_base = restrict_records(train_recs, BASE_IDS, drop_empty=True)
val_base   = restrict_records(val_recs, BASE_IDS)
test_base  = restrict_records(test_recs, BASE_IDS)
val_gt_base, test_gt_base = build_coco_gt(val_base), build_coco_gt(test_base)
print("base-training images:", len(train_base), "| base class instances:", count_table(train_base)[:3])

set_seed(SEED_FULL)
model_base = build_extreme(num_classes=1 + len(BASE_IDS), use_triplet=False).to(DEV)
trainable_base = [p for p in model_base.parameters() if p.requires_grad]
ds_base = DetDataset(train_base, train=True, color_jitter=0.15)
_dropb = len(train_base) >= 2 * BASE_BATCH
if A_USE_RFS:
    sampler_b = WeightedRandomSampler(rfs_weights(train_base, A_RFS_T, classes=BASE_IDS), num_samples=len(ds_base), replacement=True)
    loader_base = DataLoader(ds_base, batch_size=BASE_BATCH, sampler=sampler_b, num_workers=NUM_WORKERS,
                             collate_fn=collate, drop_last=_dropb)
else:
    loader_base = DataLoader(ds_base, batch_size=BASE_BATCH, shuffle=True, num_workers=NUM_WORKERS,
                             collate_fn=collate, drop_last=_dropb)

def val_fn_base(m):
    return evaluate_preds(val_base, val_gt_base, predict(m, val_loader), conf=0.5)[0]

res_base = train_loop(model_base, loader_base, val_fn_base, epochs=BASE_EPOCHS,
                      groups=[dict(params=trainable_base, lr=BASE_LR)],
                      run="B0_base_training", seed=SEED_FULL, stage="base", log_path=LOG_BASE, eval_every=1,
                      use_ema=True, patience=BASE_PATIENCE, extreme=True)
best_base = res_base["model"]
final_report("B0_base_training", SEED_FULL, best_base, kind="base_training",
             extra=dict(train_minutes=res_base["train_minutes"], trainable_params=res_base["trainable"], K=-1,
                        train_images=len(train_base)),
             val_pack=(val_base, val_gt_base), test_pack=(test_base, test_gt_base))

BASE_STATE = {k: v.detach().cpu().clone() for k, v in best_base.state_dict().items()}
torch.save(BASE_STATE, os.path.join(OUT_DIR, "ckpt", "B0_base_teacher.pth"))
del model_base, best_base, res_base, loader_base
torch.cuda.empty_cache()
print("Base model saved. Base-class results are in base_training_final_metrics.csv")

In [ ]:
# ============================ 10) EXPERIMENT B / STAGE 2: FEW-SHOT RUNNER ============================
LOG_FS = os.path.join(OUT_DIR, "epoch_log_fewshot.csv")
VARIANTS = OrderedDict([
    ("V0_TFA_cos",       dict(syn=False, unfreeze=False, kd=False, tri=False)),
    ("V1_SS",            dict(syn=True,  unfreeze=False, kd=False, tri=False)),
    ("V2_SS_unfrozen",   dict(syn=True,  unfreeze=True,  kd=False, tri=False)),
    ("V3_SS_KD",         dict(syn=True,  unfreeze=True,  kd=True,  tri=False)),
    ("V4_Extreme_RCNN",  dict(syn=True,  unfreeze=True,  kd=True,  tri=True)),
])

def run_fewshot(variant_key, seed, k=None, hp=None, report=True, save_ckpt=False, tag=""):
    k = k or K_LIST[0]
    v = VARIANTS[variant_key]
    h = dict(lam_hint=LAMBDA_HINT, lam_tri=LAMBDA_TRI, margin=TRI_MARGIN)
    h.update(hp or {})
    set_seed(seed)
    model = build_extreme(NUM_CLASSES, use_triplet=v["tri"], margin=h["margin"])
    load_base_into(model, BASE_STATE)
    model.to(DEV)
    if v["kd"]:
        teacher = copy.deepcopy(model.backbone).to(DEV).eval()
        for p in teacher.parameters():
            p.requires_grad = False
        model.set_teacher(teacher)
    groups = set_trainable(model, v["unfreeze"])
    entries = sample_kshot(train_recs, k, list(range(1, len(CLASS_NAMES) + 1)), seed)
    ds = DetDataset(entries, train=True, syn_prob=SYN_PROB if v["syn"] else 0.0, repeat=FS_REPEAT)
    loader = DataLoader(ds, batch_size=FS_BATCH, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate, drop_last=False)
    run_name = "B_" + variant_key + f"_K{k}" + tag
    res = train_loop(model, loader, val_fn_all, epochs=FS_EPOCHS, groups=groups, run=run_name, seed=seed,
                     stage="fewshot", log_path=LOG_FS, eval_every=FS_EVAL_EVERY, use_ema=False, patience=FS_PATIENCE,
                     extreme=True, lam_hint=h["lam_hint"] if v["kd"] else 0.0,
                     lam_tri=h["lam_tri"] if v["tri"] else 0.0, warmup_iters=50)
    out = dict(best_val=res["best"])
    if report:
        row = final_report(run_name, seed, res["model"], kind="fewshot",
                           extra=dict(train_minutes=res["train_minutes"], trainable_params=res["trainable"], K=k,
                                      support_instances=int(sum(len(e["labels"]) for e in entries)),
                                      lam_hint=h["lam_hint"], lam_tri=h["lam_tri"], margin=h["margin"]))
        out["row"] = row
        if SAVE_ERROR_EXAMPLES and variant_key == "V4_Extreme_RCNN" and seed == SEEDS[0] and k == K_LIST[0]:
            save_error_examples(res["model"], run_name, test_recs, row["conf_threshold"])
    if save_ckpt:
        torch.save(res["model"].state_dict(), os.path.join(OUT_DIR, "ckpt", f"{run_name}_seed{seed}.pth"))
    del model, res, loader
    if v["kd"]:
        del teacher
    torch.cuda.empty_cache()
    return out

print("few-shot runner ready; variants:", list(VARIANTS))

In [ ]:
# ============================ 11) OPTIONAL: SMALL HYPER-PARAMETER SWEEP (validation-only) ============================
# The paper does not give lambda_hint, lambda_tri or the triplet margin, so tuning them can matter.
# Selection uses the VALIDATION set only (nothing is reported on test here). Set RUN_SWEEP = True in the config cell.
if RUN_SWEEP:
    grid = [dict(lam_hint=lh, lam_tri=lt, margin=mg)
            for lh in (1.0, 10.0) for lt in (0.1, 0.5, 1.0) for mg in (0.2, 0.4)]
    sweep_rows = []
    for hp in grid:
        out = run_fewshot("V4_Extreme_RCNN", seed=SEEDS[0], hp=hp, report=False, tag="_sweep")
        sweep_rows.append(dict(hp, best_val_mAP50=out["best_val"]))
        pd.DataFrame(sweep_rows).to_csv(os.path.join(OUT_DIR, "hparam_sweep_val.csv"), index=False)
    sw = pd.DataFrame(sweep_rows).sort_values("best_val_mAP50", ascending=False)
    print(sw)
    bestrow = sw.iloc[0]
    LAMBDA_HINT, LAMBDA_TRI, TRI_MARGIN = float(bestrow["lam_hint"]), float(bestrow["lam_tri"]), float(bestrow["margin"])
    print("Using best sweep setting:", LAMBDA_HINT, LAMBDA_TRI, TRI_MARGIN)
else:
    print("Sweep skipped (RUN_SWEEP = False).")

In [ ]:
# ============================ 12) RUN ALL FEW-SHOT VARIANTS x SEEDS ============================
t_all = time.time()
for k_shot in K_LIST:
    for seed in SEEDS:
        for vk in VARIANTS:
            run_fewshot(vk, seed, k=k_shot, save_ckpt=(vk == "V4_Extreme_RCNN" and seed == SEEDS[0] and k_shot == K_LIST[0]))
print(f"\nAll few-shot runs finished in {(time.time() - t_all) / 60:.1f} min")
flush_results()

In [ ]:
# ============================ 13) FINAL TABLES: BASE, FEW-SHOT, COMPARISON ============================
df = pd.DataFrame(RESULTS)
KEYS = ["mAP50", "mAP", "mAP75", "nAP50", "nAP", "bAP50", "bAP", "precision", "recall", "f1",
        "FNR_defects", "img_defect_recall", "img_false_alarm", "AP50_no-cap", "AP50_crumbled",
        "AP50_cap", "AP50_label", "AP50_not-crumbled"]

def summarize(d):
    rows = []
    for run, g in d.groupby("run", sort=False):
        r = {"run": run, "n_seeds": len(g)}
        for k in KEYS:
            vals = g[k].astype(float).values
            r[k + "_mean"] = np.nanmean(vals) if np.isfinite(vals).any() else np.nan
            r[k + "_std"] = np.nanstd(vals, ddof=1) if (np.isfinite(vals).sum() > 1) else 0.0
        rows.append(r)
    return pd.DataFrame(rows)

# --- base training final metrics (also saved by final_report) ---
print("\n===== BASE TRAINING (stage 1, base classes only) =====")
print(df[df["kind"] == "base_training"][["run", "mAP50", "mAP", "mAP75", "bAP50", "precision", "recall", "f1",
                                          "AP50_cap", "AP50_label", "AP50_not-crumbled"]].round(4).to_string(index=False))

# --- few-shot final metrics + summary ---
fs = df[df["kind"] == "fewshot"]
if len(fs):
    fs_sum = summarize(fs)
    fs_sum.to_csv(os.path.join(OUT_DIR, "fewshot_final_metrics_summary.csv"), index=False)
    print("\n===== FEW-SHOT (K in %s) mean over seeds =====" % str(K_LIST))
    show = ["run", "n_seeds", "nAP50_mean", "nAP50_std", "bAP50_mean", "mAP50_mean", "mAP_mean", "f1_mean",
            "FNR_defects_mean", "AP50_no-cap_mean", "AP50_crumbled_mean"]
    print(fs_sum[show].round(4).to_string(index=False))

# --- comparison table: full-data vs base vs few-shot variants ---
cmp_rows = []
for run, g in df.groupby("run", sort=False):
    r = {"run": run, "kind": g["kind"].iloc[0], "seeds": len(g), "K": g["K"].iloc[0]}
    for k in ["nAP50", "bAP50", "mAP50", "mAP", "f1", "FNR_defects", "AP50_no-cap", "AP50_crumbled"]:
        vals = g[k].astype(float).values
        m = np.nanmean(vals) if np.isfinite(vals).any() else np.nan
        s = np.nanstd(vals, ddof=1) if np.isfinite(vals).sum() > 1 else 0.0
        r[k] = "-" if not np.isfinite(m) else (f"{m:.3f}" if len(g) == 1 else f"{m:.3f} +/- {s:.3f}")
    r["trainable_params_M"] = round(float(g["trainable_params"].iloc[0]) / 1e6, 2)
    r["train_minutes"] = round(float(g["train_minutes"].mean()), 1)
    cmp_rows.append(r)
comparison = pd.DataFrame(cmp_rows)
comparison.to_csv(os.path.join(OUT_DIR, "comparison_table.csv"), index=False)
print("\n===== COMPARISON (test set, confidence threshold tuned on validation) =====")
print(comparison.to_string(index=False))
flush_results()

In [ ]:
# ============================ 14) PLOTS ============================
fig_dir = os.path.join(OUT_DIR, "plots"); os.makedirs(fig_dir, exist_ok=True)

# learning curves (validation mAP50 per epoch)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, (fn, title) in zip(axes, [("epoch_log_full_data.csv", "A: full-data"), ("epoch_log_base_training.csv", "B stage 1: base"),
                                  ("epoch_log_fewshot.csv", "B stage 2: few-shot (seed %d)" % SEEDS[0])]):
    p = os.path.join(OUT_DIR, fn)
    if os.path.exists(p):
        L = pd.read_csv(p)
        if "seed" in L.columns and fn == "epoch_log_fewshot.csv":
            L = L[L["seed"] == SEEDS[0]]
        for run, g in L.groupby("run", sort=False):
            g = g.dropna(subset=["val_mAP50"])
            ax.plot(g["epoch"], g["val_mAP50"], marker="o", ms=3, label=run)
        ax.set_title(title); ax.set_xlabel("epoch"); ax.set_ylabel("val mAP50"); ax.grid(alpha=0.3); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(os.path.join(fig_dir, "learning_curves.png"), dpi=150); plt.show()

# novel AP50 comparison bars
d = pd.DataFrame(RESULTS)
d = d[d["kind"].isin(["full_data", "fewshot"])]
if len(d):
    g = d.groupby("run", sort=False)["nAP50"].agg(["mean", "std"]).fillna(0)
    plt.figure(figsize=(9, 4))
    plt.bar(range(len(g)), g["mean"], yerr=g["std"], capsize=3)
    plt.xticks(range(len(g)), g.index, rotation=30, ha="right"); plt.ylabel("novel AP50 (test)")
    plt.title("Novel-class AP50 (no-cap, crumbled)"); plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "novel_ap50_comparison.png"), dpi=150); plt.show()

# confusion matrices for A and the full Extreme R-CNN (first seed)
for key in [("A_full_data", SEED_FULL)] + [("B_V4_Extreme_RCNN_K%d" % kk, SEEDS[0]) for kk in K_LIST]:
    if key in CONFS:
        cm = CONFS[key]
        plt.figure(figsize=(6.5, 5))
        plt.imshow(cm, cmap="Blues")
        plt.xticks(range(len(CLASS_NAMES) + 1), CLASS_NAMES + ["MISSED"], rotation=45, ha="right")
        plt.yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=8)
        plt.xlabel("predicted"); plt.ylabel("ground truth"); plt.title("Confusion (test) " + key[0])
        plt.tight_layout(); plt.savefig(os.path.join(fig_dir, "confusion_%s.png" % key[0]), dpi=150); plt.show()

In [ ]:
# ============================ 15) PACKAGE OUTPUTS ============================
print("Files in", OUT_DIR)
for f in sorted(glob.glob(os.path.join(OUT_DIR, "*.csv"))):
    print("  ", os.path.basename(f), os.path.getsize(f), "bytes")
zip_path = shutil.make_archive("/kaggle/working/bottle_defect_results", "zip", OUT_DIR)
print("Zipped to", zip_path, "(download from the Output tab)")